# Ranking & Filtering

Authors:
- Ramon Casas i Luque
- Marina Velasco Barriuso
- Shaoya Zhang

>[!NOTE] Run locally so ../../data/ resolves correctly from the project root.

In this notebook we implement and compare different approaches to measure relevance.

# Part 0: Import libraries and data

>[!NOTE] `utils.py` contains some functions we wrote in part 2. They had to be slightly modified to make them work as an independent module.

In [6]:
import utils
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
import ast
from gensim.models import Word2Vec

In [7]:
# Open csv with preprocessed data
filename = "../../data/index_info.csv"
index_info, fields = utils.load_index_info(filename)

# Create tf-idf index
index, tf, df, idf = utils.create_index_tfidf(index_info, num_documents=len(index_info), fields=fields)

# Part 1: Implementation of three ranking strategies

## a. TF-IDF + cosine similarity

We already implemented this strategy in part 2. Here we reuse the functions we created to score products by TF-IDF + cosine similarity.

In [8]:
# Try one test query
products = pd.read_csv("../../data/fashion_products.csv")
query = utils.TEST_QUERIES[2]
equal_weights = np.array(len(fields)*[1/len(fields)])
ranked_docs = utils.search_tf_idf(query, index, tf, idf, fields, equal_weights)
utils.display_results(query, ranked_docs, products, top=7)

🔎 You searched for 'black sports shoes'

====================== Top 7 results out of 94 ====================== 

1 JAGUAR Football Shoes For Women  (Red, Black) | Vector | Footwear > Men's Footwear | SHOFN3EJZKBG6QUF
   sports shoes

2 Lace Up For Men  (Black) | Bone | Footwear > Men's Footwear | SHOFVXG5XR59AFTH
   Bonexy, a brand well known for its domination over the footwear indust...

3 Lace Up For Men  (Black) | Bone | Footwear > Men's Footwear | SHOFVJX2TP8FRR7P
   Bonexy, a brand well known for its domination over the footwear indust...

4 Loafers For Women  (Black) | Bone | Footwear > Men's Footwear | SHOFX2G6FHVNAHZN
   Bonexy, a brand well known for its domination over the footwear indust...

5 Loafers For Men  (Black) | Bone | Footwear > Men's Footwear | SHOFVHY8MAYUPTZF
   Bonexy, a brand well known for its domination over the footwear indust...

6 RS-7250 Running Shoes For Men  (Black) | Vector | Footwear > Men's Footwear | SHOFN3EJHXHGSGUC
   sports shoes

7 Sneakers For

## b. BM25

In this section we implement the BM25 ranking algorithm, which assigns a score $RSV_d$ to each document $d$ and then sorts them in decreasing order.

$$
RSV_d = \sum_{t \in q} \underbrace{\log \left[ \frac{N}{df_t} \right]}_{\text{IDF}} \cdot \underbrace{\frac{(k_1 + 1)tf_{td}}{k_1((1 - b) + b \times (L_d / L_{ave})) + tf_{td}}}_{\text{tf + doc. length normalization}} \cdot \underbrace{\frac{(k_3 + 1)tf_{tq}}{k_3 + tf_{tq}}}_{\text{Query term}}
$$

Calculating this score involves assigning one weight to each term in the query, and then summing all term weights. Each term weight depends on the following variables:

- $N$: number of docs in collection
- $df_t$: document frequency of term $t$ (i.e. how many docs does $t$ appear in?)
- $tf_{td}$: frequency of term $t$ in document $d$ (i.e. how many times does $t$ appear in $d$?)
- $L_d$: length of document $d$
- $L_{ave}$: average document length in the whole collection
- $k_1$: tuning parameter controlling importance of document term frequency scaling
- $b$: tuning parameter controlling importance of document length


The **query term is optional**, we can use it for long queries.
- $tf_{tq}$ : frequency of term $t$ in query $q$
- $k_3$ : tuning parameter controlling term frequency scaling of the query

>[!NOTE] We chose *not* to implement the query term, as all of our queries are relatively short (3-4 words).

**Definition of "document"**:

For us, a document will be the concatenation of all words in a the following fields of a product: `'title', 'description', 'brand', 'category', 'sub_category', 'seller', 'product_details'`

In [41]:
index_info.head(1).to_dict()

{'title': {'4hWtdShnw%3D%3D': ['men',
   'regular',
   'fit',
   'solid',
   'casual',
   'shirt']},
 'description': {'4hWtdShnw%3D%3D': []},
 'brand': {'4hWtdShnw%3D%3D': ['crock', 'cl']},
 'category': {'4hWtdShnw%3D%3D': ['cloth', 'accessori']},
 'sub_category': {'4hWtdShnw%3D%3D': ['topwear']},
 'seller': {'4hWtdShnw%3D%3D': ['crockscl']},
 'product_details': {'4hWtdShnw%3D%3D': ['1',
   'c328',
   '2',
   '38',
   'regular',
   'pure',
   'cotton',
   'full',
   'sleev',
   'solid',
   'no',
   'light',
   'blue',
   'hand',
   'wash',
   'western',
   'wear',
   'shirt',
   'india']},
 'document': {'4hWtdShnw%3D%3D': ['men',
   'regular',
   'fit',
   'solid',
   'casual',
   'shirt',
   'crock',
   'cl',
   'cloth',
   'accessori',
   'topwear',
   'crockscl',
   '1',
   'c328',
   '2',
   '38',
   'regular',
   'pure',
   'cotton',
   'full',
   'sleev',
   'solid',
   'no',
   'light',
   'blue',
   'hand',
   'wash',
   'western',
   'wear',
   'shirt',
   'india']}}

In [9]:
# Create doc_lengths: a dict where keys are product IDs
# and values are the lengths of each document
index_info["document"] = index_info[fields].apply(
    lambda row: [term for col in row if isinstance(col, list) for term in col],
    axis=1,
)
doc_lengths = dict(zip(index_info["pid"], index_info["document"].str.len()))

# Find average document length
average_length = np.mean(index_info["document"].str.len())
print(f"The average document length is {average_length:.2f} terms")

The average document length is 56.47 terms


In [10]:
# Create dictionary to store raw term frequencies of each product
# tf_td
bm25_tf = defaultdict(dict)
for pid, doc in zip(index_info["pid"], index_info["document"]): # (PID, [merged terms])
    for term, count in Counter(doc).items():
        bm25_tf[term][pid] = count

# Usage example
term = 'cotton'
pid = 'TKPFCZ9EA7H5FYZH'
print(f"Term '{term}' shows up in product '{pid}' {bm25_tf[term][pid]} times")

Term 'cotton' shows up in product 'TKPFCZ9EA7H5FYZH' 2 times


In [11]:
def bm25_score(pid, query, k1, b, doc_lengths, average_length, tf, idf):
    """_summary_
    
    Args:
        pid: product id
        query: query terms
        k1: hyper-param tuning weight of term frequency
        b: hyper-param tuning weight of document length
        doc_lengths: document length
        average_length: average document length over the collection
        tf: term frequency of the query in the document with pid
        idf: inverse document frequency of term

    Returns:
        BM25 score
    """
    rsv = 0
    for t in utils.build_terms(query):
        term1 = idf[t]
        term2 = ( (k1 + 1)*tf[t][pid] ) / ( k1 * ( (1-b)+b * (doc_lengths[pid]/average_length)) + tf[t][pid] )
        rsv += term1 * term2

    return rsv

def search_bm25(query, k1, b, doc_lengths, average_length, tf, idf):
    """#TODO document

    Args:
        query: query terms
        k1: hyper-param tuning weight of term frequency
        b: hyper-param tuning weight of document length
        doc_lengths: document length
        average_length: average document length over the collection
        tf: term frequency of the query in the document with pid
        idf: inverse document frequency of term

    Returns:
        ranked documents sorted by its BM25 scores
    """
    docs = utils.search(query, index=index)

    doc_scores = []
    for pid in docs:
        rsv = bm25_score(pid, query, k1, b, doc_lengths, average_length, tf, idf)
        doc_scores.append((rsv, pid))
    
    doc_scores.sort(reverse=True)
    result_docs = [x[1] for x in doc_scores]
    return result_docs


k1 = 1.2
b = 0.75

ranked_docs = search_bm25(query, k1, b, doc_lengths, average_length, bm25_tf, idf)

utils.display_results(query, ranked_docs, df=products, top=7)

🔎 You searched for 'black sports shoes'

====================== Top 7 results out of 94 ====================== 

1 JAGUAR Football Shoes For Women  (Red, Black) | Vector | Footwear > Men's Footwear | SHOFN3EJZKBG6QUF
   sports shoes

2 RS-7250 Running Shoes For Men  (Black) | Vector | Footwear > Men's Footwear | SHOFN3EJHXHGSGUC
   sports shoes

3 Football Shoes For Men  (Green, Black) | Vector | Footwear > Men's Footwear | SHOEWQKVXCQRCSSK
   

4 Breeze Football Shoes For Women  (Green, Black) | Vector | Footwear > Men's Footwear | SHOFFTZNUFSFGGMR
   

5 Basketball Shoes For Men  (Green, Black) | Vector | Footwear > Men's Footwear | SHOEZHDKSGJCSUEJ
   

6 Cricket Shoes For Women  (White, Black) | Vector | Footwear > Men's Footwear | SHOEWQKVYWWZ2CZK
   

7 Cricket Shoes For Women  (White, Black) | Vector | Footwear > Men's Footwear | SHOEWQKV5PHERHXN
   



## c. Our score

In this section we create a custom score to rank our products.

Hint: look into numerical fields

Ideas:
- penalize products without an image --> we can attribute it to product info completeness
- reward products with higher rating
- reward products with discount
- Coverage percentage of query terms in title --> in lab1 we have concluded that title is more important than other fields, so we can apply this, and we can also analyze its pro and cons



Our algorithm: BM25 + PICTURE + RATING + DISCOUNT + TITLE



In [12]:
# 1. penalize with products withoug an image url

def has_image_url(value):
    """"
    :param value: value to verify
    :returns: `True` if `value` is an image url (starts with https:// or http://) 
    """

    if isinstance(value, str):
        try:
            value = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return False
        
    return isinstance(value, list) and any(
        isinstance(url, str)
        and url.strip().startswith(("https://", "http://"))
        for url in value
    )

# Verification
products["has_image_url"] = products["images"].apply(has_image_url)
print(products["has_image_url"].value_counts())

has_image_url
True     27308
False      772
Name: count, dtype: int64


In [13]:
# 4. Title coverage
def title_coverage(query, title_terms):
    """Calculate a title coverage multiplier, returning 1.0 for missing values
    
    :param query: query terms
    :param title_terms: product titles after preprocessing
    :returns: query coverage factor: % of words in the query that are in the product's title
    """
    query_terms = set(utils.build_terms(query))
    
    if len(query_terms) > 0:
        return len(query_terms.intersection(title_terms)) / len(query_terms)
    else:
        raise ValueError("Query is empty")

$$\text{Our score}_i = (w_0 · \text{BM25}_i) + (w_1 · \text{has image}_i) + (w_2 · \tilde r_i) + (w_3 · tc_i) + (w_4 · d_i) $$

Where:
- $\text{BM25}_i$ is the BM25 score of product $i$
- $\text{has image}_i)$ is 1 if product $i$ has an image, 0 otherwise
- $\tilde r_i$ is the normalized rating of product $i$
  - if it is NaN we assume it has a rating of 2 $r_i = 2$. The reason for this is that a product with no reviews is better than a product with very bad reviews (1) but worse than a product with mediocre reviews.
- $tc_i$ is the title coverage (%) of the query for product $i$
- $d_i$ is the discount of product $i$ (if it is NaN we assume it has no discount, $d_i = 0$)

In [14]:
def our_score(pid, query, k1, b, doc_lengths, average_length, tf, idf, products_df, weights):
    """_summary_
    
    Args:
        pid: product id
        query: query terms
        k1: hyper-param tuning weight of term frequency
        b: hyper-param tuning weight of document length
        doc_lengths: document length
        average_length: average document length over the collection
        tf: term frequency of the query in the document with pid
        idf: inverse document frequency of term

    Returns:
        BM25 score
    """
    if len(weights) != 5:
        raise ValueError(f"Weights must be an array with 5 elements but it has {len(weights)}")

    bm25 = 0
    for t in utils.build_terms(query):
        term1 = idf[t]
        term2 = ( (k1 + 1)*tf[t][pid] ) / ( k1 * ( (1-b)+b * (doc_lengths[pid]/average_length)) + tf[t][pid] )
        bm25 += term1 * term2

    product = products_df[products_df['pid'] == pid]

    normalized_rating = np.clip(product['average_rating'].values[0], 0, 5) / 5
    normalized_rating = 2.0 / 5 if pd.isna(normalized_rating) else normalized_rating

    discount = product['discount'].values[0]
    discount = 0 if pd.isna(discount) else discount

    our_score = (weights[0] * bm25 +
                 weights[1] * has_image_url(product['images'].values[0]) +
                 weights[2] * normalized_rating +
                 weights[3] * title_coverage(query, utils.build_terms(product['title'].values[0])) +
                 weights[4] * discount
                 )

    return our_score

def search_our_score(query, k1, b, doc_lengths, average_length, tf, idf, products_df, weights):
    """#TODO document

    Args:
        query: query terms
        k1: hyper-param tuning weight of term frequency
        b: hyper-param tuning weight of document length
        doc_lengths: document length
        average_length: average document length over the collection
        tf: term frequency of the query in the document with pid
        idf: inverse document frequency of term

    Returns:
        ranked documents sorted by its BM25 scores
    """
    docs = utils.search(query, index=index)

    doc_scores = []
    for pid in docs:
        rsv = our_score(pid, query, k1, b, doc_lengths, average_length, tf, idf, products_df, weights)
        doc_scores.append((rsv, pid))
    
    doc_scores.sort(reverse=True)
    result_docs = [x[1] for x in doc_scores]
    return result_docs


k1 = 1.2
b = 0.75
query = 'black sports shoes'
weights = [1, 0.2, 0.3, 0.8, 0.1]

ranked_docs = search_our_score(query, k1, b, doc_lengths, average_length, bm25_tf, idf, products, weights)

utils.display_results(query, ranked_docs, df=products, top=7)

🔎 You searched for 'black sports shoes'

====================== Top 7 results out of 94 ====================== 

1 JAGUAR Football Shoes For Women  (Red, Black) | Vector | Footwear > Men's Footwear | SHOFN3EJZKBG6QUF
   sports shoes

2 RS-7250 Running Shoes For Men  (Black) | Vector | Footwear > Men's Footwear | SHOFN3EJHXHGSGUC
   sports shoes

3 Football Shoes For Men  (Green, Black) | Vector | Footwear > Men's Footwear | SHOEWQKVXCQRCSSK
   

4 Breeze Football Shoes For Women  (Green, Black) | Vector | Footwear > Men's Footwear | SHOFFTZNUFSFGGMR
   

5 Sports Shoes For Women  (Black, Yellow) | Vector | Footwear > Men's Footwear | SHOE7TZFE2TCNPH5
   Vector X Is One Of The Established Sports Brands Who Provide Stylish A...

6 Cricket Shoes For Women  (White, Black) | Vector | Footwear > Men's Footwear | SHOEWQKV5PHERHXN
   

7 Basketball Shoes For Men  (Green, Black) | Vector | Footwear > Men's Footwear | SHOEZHDKSGJCSUEJ
   



## Comparison of a. b. and c.

Explain how the ranking differs when using TF-IDF and BM25, and think about the pros 
and  cons  of  using  each  of  them.  Regarding your own score, justify the choice of the 
score (pros and cons). HINT: Look into numerical fields that each record has to build your 
score.  


Now, to compare the three strategies we:
1. **Run our 5 test queries with each method and keep the top 20 results.**
2. **Build a pool of results and export it to label by hand**.
   - The pool of results is the union of the top 20 of the three methods for each query. Each (query, product) pair is labeled only once, even if several methods retrieved it. Products are sorted by pid so we don't know which method ranked them higher when labeling.
   - In this step we reuse labels from Part 2 (`test_queries_results.csv`) so that we only need to label the new products.
3. Compute **NDCG@20** for every method. The ideal DCG is computed from the whole pool, so a method that misses relevant products found by the others is penalized.

In [15]:
# 1. Run our 5 test queries with each method and keep the top 20 results.
TOP_N = 20
POOL_CSV = "../../data/test_queries_pool_labels.csv"
PART2_LABELS_CSV = "../../data/test_queries_results.csv"

test_queries = {i: q for i, q in enumerate(utils.TEST_QUERIES, start=1)}

# Each method: function(query) -> list of ranked pids
search_functions = {
    "TF-IDF": lambda q: utils.search_tf_idf(q, index, tf, idf, fields, equal_weights),
    "BM25": lambda q: search_bm25(q, k1, b, doc_lengths, average_length, bm25_tf, idf),
    "Our score": lambda q: search_our_score(q, k1, b, doc_lengths, average_length, bm25_tf, idf, products, weights),
}

rankings = utils.get_rankings(test_queries, search_functions, top_n=TOP_N)

In [16]:
# 2. Export the pool to label by hand (never overwrites an existing file)
pool_df = utils.export_pooled_rankings(
    POOL_CSV, test_queries, rankings, products, previous_labels_path=PART2_LABELS_CSV
)

../../data/test_queries_pool_labels.csv already exists, nothing exported. Delete it manually to generate it again.


Now we open `data/test_queries_pool_labels.csv` and fill in the empty `labels` cells with `1` (relevant) or `0` (non-relevant). Once all products have a relevance label, we open the file.

In [17]:
# 3. Read the labels and compute NDCG@20 of each method
labels_df = utils.load_labels(POOL_CSV)
ndcg = utils.evaluate_rankings_ndcg(test_queries, rankings, labels_df, k=TOP_N)

with pd.option_context("display.float_format", "{:.3f}".format):
    display(ndcg)

,TF-IDF,BM25,Our score
1: men's cotton checkered shirt,1.000,1.000,1.000
2: women casual shirt linen,1.000,1.000,1.000
3: black sports shoes,0.682,1.000,1.000
4: soft jacket for women,0.371,0.193,0.193
5: grey hoodie with pockets,0.969,0.935,0.939
mean,0.804,0.826,0.826


In [18]:
# How similar are the top 20 results of the methods? (mean Jaccard overlap over the 5 queries)
with pd.option_context("display.float_format", "{:.2f}".format):
    display(utils.rankings_overlap(rankings, k=TOP_N))

,TF-IDF,BM25,Our score
TF-IDF,1.00,0.69,0.64
BM25,0.69,1.00,0.91
Our score,0.64,0.91,1.00


In short, our score behaves similar to BM25. It has similar evaluation metrics across the five queries, with slightly lower values for queries 4 and 5. 

The fact that they are so similar makes sense -- BM25 is one of the components that makes up our score, and it is the one with the largest value (1 vs. 0.2 to 0.8 for the other components).

Our goal was to improve on BM25 but we clearly failed at doing so. However, we must also note that when we evaluate whether a product is relevant or not for a specific query we didn't see whether the product has an image or not, or the quality of the product, which are relevant for our score. Ideally, when we evaluate the relevance of a product we should take into account more factors other than the title and description. To to this effectively, we'd need a UI that showed the product's image, as well as the user rating and discount, and pay workers to label the product as relevant (1) or not relevant (0).

In [19]:
# TODO: decide if we discuss the differences qualitatively or not
#  # Side-by-side top 10 for a query (title + manual label), to discuss the differences qualitatively
# query_id = 5
# top = 10

# titles = products.drop_duplicates("pid").set_index("pid")["title"]
# query_labels = labels_df[labels_df["query_id"] == query_id].set_index("pid")["labels"]

# print(f"Query {query_id}: '{test_queries[query_id]}'")
# pd.concat(
#     {
#         m: pd.DataFrame({
#             "title": titles.reindex(r[query_id][:top]).to_list(),
#             "label": query_labels.reindex(r[query_id][:top]).astype("Int64").to_list(),
#         })
#         for m, r in rankings.items()
#     },
#     axis=1,
# ).set_axis(range(1, top + 1))

## Part 2: Implementation of word2vec + cosine ranking score

In this section, we ranking using a word2vec model for documents plus a cosine ranking score.  
To represent a piece of text using word2vec, we create a single vector that represents 
the entire text. This vector has the same number of dimensions as the word vectors and 
is calculated by averaging the vectors of all words in the text.

In [20]:
# Reference: https://www.geeksforgeeks.org/nlp/word2vec-with-gensim/

# Build the corpus:
# Each sentence is one product's title or description
cols = ['title', 'description']
sentences = []
for col in cols:
    l = index_info[col].to_list()
    sentences.extend(l)

# This should be 2 * num_products
print(f"Got a total of {len(sentences)} sentences")

model = Word2Vec(sentences=sentences, vector_size=256, window=5, min_count=1, workers=4, sg=0)
model.save("word2vec.model")

# 1. Convert all products to vectors using Word2Vec approach
# Each product becomes a vector with 256 components
index_info = index_info.set_index('pid')
product_terms = (index_info['title'] + index_info['description']) # Extract product terms
doc_vecs = product_terms.apply(
    lambda terms: model.wv.get_mean_vector( # Product vector is the average of all term vectors
    terms, 
    pre_normalize=False,
    post_normalize=True # Normalize so we dont normalize in cosine similarity
    )
)
print(f"A total of {len(doc_vecs)} products were converted to vectors")

Got a total of 56160 sentences
A total of 28080 products were converted to vectors


Here is an example of how product with pid `'ARME8P6GFSXGE9AW'` is transformed from product to terms to vector.

In [21]:
pid = 'ARME8P6GFSXGE9AW'
product = products.set_index('pid').loc[pid]
print(f"Example: product with pid {pid}")
display(product[['title', 'description', 'seller', 'average_rating', 'product_details']])

print("Product represented as terms (title + description):")
print(f"\t {product_terms.loc[pid]}")

print("Product represented as a vector:")
print(f"\t{doc_vecs.loc[pid][0:5]}... ({len(doc_vecs.loc[pid])} dimensions)")

Example: product with pid ARME8P6GFSXGE9AW


title              Welwear suncool1008 Nylon Arm Warmer  (Multico...
description        Tatto Sleeve Protect From Uv Raditaion And Als...
seller                                                  NAVKAR CRAFT
average_rating                                                   2.0
product_details    [{'Pack of': '2'}, {'Sales Package': '2 Arm Wa...
Name: ARME8P6GFSXGE9AW, dtype: object

Product represented as terms (title + description):
	 ['welwear', 'suncool1008', 'nylon', 'arm', 'warmer', 'multicolor', 'tatto', 'sleev', 'protect', 'uv', 'raditaion', 'also', 'provid', 'sooth', 'feel', 'hot', 'condit']
Product represented as a vector:
	[-0.06231103  0.00172357  0.0177572   0.05171121 -0.09432493]... (256 dimensions)


Now, we return a top-20 list of documents for each of our 5 test queries using search and word2vec + cosine similarity ranking. 

In [27]:
def search_word2vec(query, model, index, doc_vecs):
    # vectorize query
    query_terms = utils.build_terms(query)
    query_vec = model.wv.get_mean_vector(
        query_terms,
        pre_normalize = False,
        post_normalize = True
    )
    
    # Return conjunctive query results 
    docs = set(utils.search(query, index))
    
    # Assign each returned relevant document its score
    cosine_scores = doc_vecs.apply(
        lambda doc_vec: np.dot(query_vec, doc_vec)
    )
    
    # Assign each returned relevant document its score
    doc_scores = [
        (score, pid)
        for score, pid in list(zip(cosine_scores, index_info.index))
        if pid in docs
    ]
    doc_scores.sort(reverse=True)
    ranked_docs = [pid for _, pid in doc_scores]

    return ranked_docs

# 2. Iterate the 5 queries and show results
for query in utils.TEST_QUERIES:
    ranked_docs = search_word2vec(query, model, index, doc_vecs)  
    
    utils.display_results(query, ranked_docs, df=products, top=20)

🔎 You searched for 'men's cotton checkered shirt'

====================== Top 20 results out of 401 ====================== 

1 Men Slim Fit Checkered Spread Collar Casual Shirt | Adam Park | Clothing and Accessories > Topwear | SHTFEW5TGWFDUDMH
   Blue Checkered Cotton Men Shirts

2 Men Slim Fit Checkered Club Collar Casual Shirt | Adam Park | Clothing and Accessories > Topwear | SHTFEUZ8CUGZXD54
   South India Shopping Mall Pink Checkered Cotton Men Shirts

3 Men Slim Fit Checkered Club Collar Casual Shirt | Adam Park | Clothing and Accessories > Topwear | SHTFEUZGSGGMYYVK
   South India Shopping Mall Green Checkered Cotton Men Shirts

4 Men Slim Fit Checkered Club Collar Casual Shirt | Adam Park | Clothing and Accessories > Topwear | SHTFEUZF9UT7ZRDN
   South India Shopping Mall Navy Checkered Cotton Men Shirts

5 Men Slim Fit Checkered Club Collar Casual Shirt | Adam Park | Clothing and Accessories > Topwear | SHTFEQJQ77NQZFX9
   South India Shopping Mall Blue Checkered Cotton Men S

In [34]:
# Side-by-side top 20 for a query (title + description), to discuss the differences qualitatively
query_id = 5
top = 20

# Each method: function(query) -> list of ranked pids
search_functions = {
    "TF-IDF": lambda q: utils.search_tf_idf(q, index, tf, idf, fields, equal_weights),
    "BM25": lambda q: search_bm25(q, k1, b, doc_lengths, average_length, bm25_tf, idf),
    "Our score": lambda q: search_our_score(q, k1, b, doc_lengths, average_length, bm25_tf, idf, products, weights),
    "Word2Vec": lambda q: search_word2vec(q, model, index, doc_vecs)  ,
}

rankings = utils.get_rankings(test_queries, search_functions, top_n=TOP_N)
titles = products.drop_duplicates("pid").set_index("pid")["title"]
descriptions = products.drop_duplicates("pid").set_index("pid")["description"]
query_labels = labels_df[labels_df["query_id"] == query_id].set_index("pid")["labels"]

print(f"Query {query_id}: '{test_queries[query_id]}'")
pd.concat(
    {
        m: pd.DataFrame({
            "title": titles.reindex(r[query_id][:top]).to_list(),
            "description": descriptions.reindex(r[query_id][:top]).to_list(),
        })
        for m, r in rankings.items()
    },
    axis=1,
).set_axis(range(1, top + 1))

Query 5: 'grey hoodie with pockets'


TF-IDF  \
                                          title   
1            Full Sleeve Printed Men Sweatshirt   
2            Full Sleeve Solid Women Sweatshirt   
3          Full Sleeve Printed Women Sweatshirt   
4            Full Sleeve Solid Women Sweatshirt   
5   Full Sleeve Self Design Women Sports Jacket   
6              Full Sleeve Solid Men Sweatshirt   
7              Full Sleeve Solid Men Sweatshirt   
8              Half Sleeve Solid Men Sweatshirt   
9            Full Sleeve Printed Men Sweatshirt   
10      Typography Men Hooded Neck Grey T-Shirt   
11         Full Sleeve Printed Women Sweatshirt   
12             Full Sleeve Solid Men Sweatshirt   
13        Full Sleeve Solid Women Sports Jacket   
14          Full Sleeve Solid Men Casual Jacket   
15          Full Sleeve Solid Men Casual Jacket   
16           Full Sleeve Printed Men Sweatshirt   
17           Full Sleeve Printed Men Sweatshirt   
18       Full Sleeve Applique Men Casual Jacket   
19     Self Design Men Hooded Neck Grey T-Shirt   
20             Full Sleeve Solid Men Sweatshirt   

                                                       \
                                          description   
1   "Revitalize your clothing with the awesome col...   
2        Grey melange front open-hoodie with pockets.   
3   PRODUCT STORY  Be iconic. This new essential w...   
4   Full-zip fleece hoodie provides comfort  Brush...   
5                                     ESS FZ Hoody FL   
6   PRODUCT STORY   Amplify your look with this ho...   
7   PRODUCT STORY   The Rebel Full Zip Men's Hoodi...   
8   PRODUCT STORYTake your workouts to the next le...   
9   PRODUCT STORY   Equip yourself with style pack...   
10  An all season long top, sleeveless to give ext...   
11                                                NaN   
12  From squat presses to shuttle runs, this men's...   
13  From squat presses to shuttle runs, this women...   
14                                                NaN   
15                                                NaN   
16  PRODUCT STORY   Catch all the best attention i...   
17  PRODUCT STORY   Strut your stuff with street-r...   
18                                                NaN   
19  Tees Collection T-Shirts & Hoodies are made of...   
20  Mens Hoodie made in Cotton Fleece with brushed...   

                                           BM25  \
                                          title   
1            Full Sleeve Solid Women Sweatshirt   
2            Full Sleeve Printed Men Sweatshirt   
3            Full Sleeve Solid Women Sweatshirt   
4   Full Sleeve Self Design Women Sports Jacket   
5              Full Sleeve Solid Men Sweatshirt   
6           Full Sleeve Solid Men Casual Jacket   
7       Typography Men Hooded Neck Grey T-Shirt   
8              Full Sleeve Solid Men Sweatshirt   
9          Full Sleeve Printed Women Sweatshirt   
10          Full Sleeve Solid Men Casual Jacket   
11         Full Sleeve Printed Women Sweatshirt   
12             Half Sleeve Solid Men Sweatshirt   
13           Full Sleeve Printed Men Sweatshirt   
14        Full Sleeve Solid Women Sports Jacket   
15       Full Sleeve Applique Men Casual Jacket   
16             Full Sleeve Solid Men Sweatshirt   
17             Full Sleeve Solid Men Sweatshirt   
18           Full Sleeve Printed Men Sweatshirt   
19     Self Design Men Hooded Neck Grey T-Shirt   
20           Full Sleeve Printed Men Sweatshirt   

                                                       \
                                          description   
1        Grey melange front open-hoodie with pockets.   
2   "Revitalize your clothing with the awesome col...   
3   Full-zip fleece hoodie provides comfort  Brush...   
4                                     ESS FZ Hoody FL   
5   PRODUCT STORY   Amplify your look with this ho...   
6                                                 NaN   
7   An all season long top, sleeveless to give ext...   
8   PRODUCT STORY   The Reb

# Part 3: Alternative representations of documents

(HINT - what about Doc2vec? Sentence2vec? What are the pros and cons

- Doc2Vec already exists
- search for the pros and cons of each
- we could try applying doc2vec to our corpus and finding differences in ranking (https://radimrehurek.com/gensim/auto_examples/tutorials/run_doc2vec_lee.html)